In [1]:

# Demo: insert_kmers replaces a zero-length marker <bc/> with <bc>[kmer]</bc>
import poolparty as pp
pp.init()
template_pool = (
    pp.from_seq('TCCCGACT<cre>GGAAAGCGGGC<misc/>AGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')
    .named('template_pool')
)

wt_pool = (
    template_pool
    .repeat_states(3, seq_name_prefix='wt.v')
    .named('wt_pool')
)

del_pool = (
    template_pool
    .deletion_scan('cre', deletion_length=5, positions=slice(None, None, 5), mode='hybrid', num_hybrid_states=3, seq_name_prefix='del_')
    .repeat_states(3, seq_name_prefix='v')
    .named('del_pool')
)

shuf_pool = (
    template_pool
    .shuffle_seq('cre', mode='hybrid', num_hybrid_states=5, seq_name_prefix='shuff_')
    .repeat_states(3, seq_name_prefix='v')
    .named('shuf_pool')
)

mut_pool = (
    template_pool
    .mutagenize('cre', mutation_rate=0.1, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')
    .repeat_states(3, seq_name_prefix='v')
    .named('mut_rate_pool')
)

site_pool = pp.from_seqs(['AAAAA', 'CCCCC'], mode='sequential', seq_name_prefix='site_')

repl_pool = (
    template_pool
    .replacement_scan('cre', ins_pool=site_pool, positions=slice(None, None, 5), mode='sequential', seq_name_prefix='repl_')
    .repeat_states(3, seq_name_prefix='v')
    .named('repl_pool')
)

combo_pool = (
    pp.stack([
        wt_pool, 
        shuf_pool,
        mut_pool,
        del_pool,
        repl_pool,
    ])
    .insert_kmers('bc', length=5)
    .named('combo_pool')
    .clear_markers()
    .upper()
    .print_library()
)

pool[17]: seq_length=None, num_states=102
state  name              seq
    0  wt.v0             TCCCGACTGGAAAGCGGGCAGTGAGCACACAGGAAAATTACGGTGGCCAGATCGGA
    1  wt.v1             TCCCGACTGGAAAGCGGGCAGTGAGCACACAGGAAAATTACGGAAAATAGATCGGA
    2  wt.v2             TCCCGACTGGAAAGCGGGCAGTGAGCACACAGGAAAATTACGGGTGGTAGATCGGA
    3  shuff_0.v0        TCCCGACTGCCTGAGGCGCGAAAGGGCAAGAAAAGAATTACGGGGGGTAGATCGGA
    4  shuff_0.v1        TCCCGACTGCCTGAGGCGCGAAAGGGCAAGAAAAGAATTACGGCTGACAGATCGGA
    5  shuff_0.v2        TCCCGACTGCCTGAGGCGCGAAAGGGCAAGAAAAGAATTACGGTGATGAGATCGGA
    6  shuff_1.v0        TCCCGACTCGAAGAAGGGGAGGGTAACAAGAACCCGATTACGGTAATAAGATCGGA
    7  shuff_1.v1        TCCCGACTCGAAGAAGGGGAGGGTAACAAGAACCCGATTACGGGACCCAGATCGGA
    8  shuff_1.v2        TCCCGACTCGAAGAAGGGGAGGGTAACAAGAACCCGATTACGGCAAAAAGATCGGA
    9  shuff_2.v0        TCCCGACTGGCAGAGAAGACCGGGACGCGATAGAAAATTACGGGGGCGAGATCGGA
   10  shuff_2.v1        TCCCGACTGGCAGAGAAGACCGGGACGCGATAGAAAATTACGGTCCTTAGATCGGA
   11  shuff_2.v2        TC

In [2]:
combo_pool.print_dag(show_pools=True)

pool[17] (pool, n=102)
└── op[17]:upper [mode=fixed, n=1]
    └── pool[16] (pool, n=102)
        └── op[16]:fixed [mode=fixed, n=1]
            └── pool[15].combo_pool (pool, n=102)
                └── op[15]:get_kmers [mode=random, n=1]
                    └── pool[14] (pool, n=102)
                        └── op[14]:stack [mode=fixed, n=1]
                            ├── pool[1].wt_pool (pool, n=3)
                            │   └── op[1]:repeat [mode=sequential, n=3]
                            │       └── pool[0].template_pool (pool, n=1)
                            │           └── op[0]:from_seq [mode=fixed, n=1]
                            ├── pool[6].shuf_pool (pool, n=15)
                            │   └── op[6]:repeat [mode=sequential, n=3]
                            │       └── pool[5] (pool, n=5)
                            │           └── op[5]:shuffle_seq [mode=hybrid, n=5]
                            │               └── pool[0].template_pool (pool, n=1)
               

Pool(id=17, name='pool[17]', op='op[17]:upper', num_states=102)